# Instacart: EDA and leakage-safe chronological split

This notebook establishes the experimental foundation; it does **not** build candidates or models. To keep order `n` genuinely untouched, target-dependent EDA uses the development target `n-1` and history through `n-2`. `order_products__train` is intentionally not loaded.

## Protocol

| Stage | Target basket | History | Purpose |
|---|---:|---:|---|
| development / ranker | `n-1` | `<= n-2` | EDA, feature development, validation |
| final locked test | `n` | `<= n-1` | one evaluation after decisions are frozen |

A repeat item has appeared in the user's visible history; an explore item has not. `add_to_cart_order` is the user's cart-add sequence, not recommendation rank.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the repository root or notebooks/.')
sys.path.insert(0, str(ROOT))

from src.data import load_instacart, resolve_data_dir
from src.evaluation import (
    assert_no_target_in_history,
    build_evaluation_examples,
    classify_repeat_explore,
    history_orders,
    target_items,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)
DATA_DIR = resolve_data_dir()
DATA_DIR

In [ ]:
# Large tables are loaded once. Final labels are intentionally absent.
data = load_instacart(
    DATA_DIR,
    tables=('orders', 'order_products__prior', 'products', 'aisles', 'departments'),
)
orders = data['orders']
prior_items = data['order_products__prior']
products = data['products']
aisles = data['aisles']
departments = data['departments']

examples = build_evaluation_examples(orders)
ranker_history_orders = history_orders(orders, examples, 'ranker')
assert_no_target_in_history(ranker_history_orders, examples, 'ranker')
assert ranker_history_orders['eval_set'].astype('string').eq('prior').all()
assert (
    ranker_history_orders.merge(
        examples[['user_id', 'ranker_history_max_order_number']],
        on='user_id', validate='many_to_one'
    )['order_number']
    <= ranker_history_orders.merge(
        examples[['user_id', 'ranker_history_max_order_number']],
        on='user_id', validate='many_to_one'
    )['ranker_history_max_order_number']
).all()

print(f'Evaluation users: {len(examples):,}')
examples.head()

## Users and orders

In [ ]:
orders_per_user = orders.groupby('user_id', observed=True).size()
prior_basket_sizes = prior_items.groupby('order_id', observed=True).size()
summary = pd.Series({
    'users': orders['user_id'].nunique(),
    'orders': orders['order_id'].nunique(),
    'evaluation_users': len(examples),
    'median_orders_per_user': orders_per_user.median(),
    'median_prior_basket_size': prior_basket_sizes.median(),
})
summary.to_frame('value')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.histplot(orders_per_user, bins=50, ax=axes[0, 0])
axes[0, 0].set(title='Orders per user', xlabel='number of orders')
sns.histplot(prior_basket_sizes.clip(upper=50), bins=50, ax=axes[0, 1])
axes[0, 1].set(title='Prior basket size (clipped at 50)', xlabel='items')
sns.histplot(orders['days_since_prior_order'].dropna(), bins=31, ax=axes[0, 2])
axes[0, 2].set(title='Days since prior order', xlabel='days')
orders['order_dow'].value_counts(normalize=True).sort_index().plot.bar(ax=axes[1, 0])
axes[1, 0].set(title='Order day of week', xlabel='day', ylabel='share')
orders['order_hour_of_day'].value_counts(normalize=True).sort_index().plot.bar(ax=axes[1, 1])
axes[1, 1].set(title='Order hour', xlabel='hour', ylabel='share')
sns.ecdfplot(prior_basket_sizes, ax=axes[1, 2])
axes[1, 2].set(title='Basket-size ECDF', xlabel='items')
fig.tight_layout()

## Items, taxonomy, long tail, and reorder rate

In [ ]:
item_stats = (
    prior_items.groupby('product_id', observed=True)
    .agg(purchases=('order_id', 'size'), reorder_rate=('reordered', 'mean'))
    .sort_values('purchases', ascending=False)
)
taxonomy_summary = pd.Series({
    'products': products['product_id'].nunique(),
    'aisles': aisles['aisle_id'].nunique(),
    'departments': departments['department_id'].nunique(),
    'overall_prior_reorder_rate': prior_items['reordered'].mean(),
})
display(taxonomy_summary.to_frame('value'))
display(products.merge(item_stats, on='product_id').nlargest(10, 'purchases')[
    ['product_name', 'purchases', 'reorder_rate']
])

In [ ]:
rank = np.arange(1, len(item_stats) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(rank, item_stats['purchases'].to_numpy())
axes[0].set(title='Product popularity long tail', xlabel='popularity rank', ylabel='purchases')
sns.histplot(item_stats['reorder_rate'], bins=40, weights=item_stats['purchases'], ax=axes[1])
axes[1].set(title='Purchase-weighted product reorder rate', xlabel='reorder rate')
fig.tight_layout()

## Repeat vs explore on the development target

The target is order `n-1`, whose rows live in `order_products__prior`. Its own products must be excluded from the history before repeat/explore membership is computed.

In [ ]:
# Keep only the columns needed for membership to control peak memory.
history_pairs = (
    ranker_history_orders[['order_id', 'user_id']]
    .merge(
        prior_items[['order_id', 'product_id']],
        on='order_id', how='inner', validate='one_to_many'
    )[['user_id', 'product_id']]
)
ranker_targets = target_items(
    prior_items,
    pd.DataFrame(),  # unused for the ranker stage
    examples,
    'ranker',
)
classified = classify_repeat_explore(history_pairs, ranker_targets)

item_type_share = classified['item_type'].value_counts(normalize=True).rename('share')
item_type_share.to_frame()

In [ ]:
user_repeat = (
    classified.assign(is_repeat=classified['item_type'].eq('repeat'))
    .groupby('user_id', observed=True)
    .agg(target_basket_size=('product_id', 'size'), repeat_ratio=('is_repeat', 'mean'))
    .reset_index()
)
target_context = orders.loc[
    orders['order_id'].isin(examples['ranker_target_order_id']),
    ['order_id', 'days_since_prior_order'],
]
user_repeat = (
    user_repeat
    .merge(
        examples[['user_id', 'ranker_target_order_id', 'ranker_history_max_order_number']],
        on='user_id', validate='one_to_one'
    )
    .merge(
        target_context, left_on='ranker_target_order_id', right_on='order_id',
        validate='one_to_one'
    )
    .rename(columns={'ranker_history_max_order_number': 'history_length'})
)
user_repeat[['repeat_ratio', 'history_length', 'target_basket_size', 'days_since_prior_order']].describe()

In [ ]:
def quantile_profile(frame, column, q=10):
    bins = pd.qcut(frame[column], q=q, duplicates='drop')
    return frame.groupby(bins, observed=True).agg(
        x=(column, 'median'), repeat_ratio=('repeat_ratio', 'mean'), users=('user_id', 'size')
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(user_repeat['repeat_ratio'], bins=21, ax=axes[0, 0])
axes[0, 0].set(title='Repeat ratio across users', xlabel='repeat share in target basket')
for ax, column, title in [
    (axes[0, 1], 'history_length', 'Repeat ratio vs history length'),
    (axes[1, 0], 'target_basket_size', 'Repeat ratio vs target-basket size'),
    (axes[1, 1], 'days_since_prior_order', 'Repeat ratio vs order interval'),
]:
    profile = quantile_profile(user_repeat.dropna(subset=[column]), column)
    sns.lineplot(data=profile, x='x', y='repeat_ratio', marker='o', ax=ax)
    ax.set(title=title, xlabel=column, ylabel='mean repeat ratio')
fig.tight_layout()

## Split audit and conclusions

The cells below report identities and cutoffs rather than model results. Final labels remain unopened.

In [ ]:
split_audit = pd.Series({
    'evaluation_users': examples['user_id'].nunique(),
    'ranker_target_orders': examples['ranker_target_order_id'].nunique(),
    'ranker_history_orders': ranker_history_orders['order_id'].nunique(),
    'target_history_order_overlap': len(
        set(examples['ranker_target_order_id']).intersection(ranker_history_orders['order_id'])
    ),
    'test_eval_users_used': 0,
    'order_products__train_rows_loaded': 0,
})
assert split_audit['target_history_order_overlap'] == 0
display(split_audit.to_frame('value'))
print('Development repeat share:', f"{item_type_share.get('repeat', 0):.3%}")
print('Development explore share:', f"{item_type_share.get('explore', 0):.3%}")